# 跑通第一个通信测试

这一节用 2 个 rank 跑一次 ReduceScatter，并用 Checker 校验任务图是否正确。

跑完你会看到 `All Success`——这就是本课"跑通了"的判定标准。

## 本节学习大纲

- 一次仿真的四个步骤
- 启动仿真环境（后台常驻）
- 建立通信域
- 跑一次 ReduceScatter
- 看数值校验结果，理解为什么它显示 failed
- 跑 Checker，这才是真正的验收
- 导出结果（下载到本地）
- 遇到问题先看这里

## 1. 一次仿真的四个步骤

| 步骤 | 做什么 | 产物 |
| :--- | :--- | :--- |
| 1 启动仿真环境 | `hccl-vm start <拓扑>` | 生成集群拓扑，起一个常驻服务 |
| 2 建立通信域 | `hccl-vm mock-comm 112` | 生成 `data/ranktable.json`，rank 与 server 建立映射 |
| 3 跑算例 | `mpirun ... reduce_scatter_test` | 算子执行，记录任务与数据 |
| 4 跑校验 | `hccl-vm plugin run @checker` | 检查任务依赖、内存访问、同步关系 |

第 2 步不能省。没有通信域，第 3 步的进程找不到自己属于哪个 server，会直接失败。

## 2. 启动仿真环境

`start` 是一个**常驻进程**，会一直监听等待后续命令。在 Notebook 里直接执行它，这个单元会永远不返回。

所以用 `tmux` 把它放到后台会话里，之后的命令通过 `tmux send-keys` 送进这个会话。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux kill-server 2>/dev/null
# kill-server 是异步的：等旧服务端真正退出，否则 new-session 会撞上"正在死的 server"
for i in $(seq 1 20); do
    tmux list-sessions >/dev/null 2>&1 || break
    sleep 0.2
done
for i in 1 2 3; do
    tmux new-session -d -s hvm -x 200 -y 50 "$PACK/run_hccl_vm.sh start $CLUSTER_YAML 2>&1 | tee /tmp/hvm_start.log" && break
    sleep 1
done
if ! tmux has-session -t hvm 2>/dev/null; then
    echo "!! session not started, start output below:"
    tail -20 /tmp/hvm_start.log 2>/dev/null || true
fi

for i in $(seq 1 40); do
    tmux capture-pane -pt hvm 2>/dev/null | grep -q "listening" && break
    sleep 1
done

tmux capture-pane -pt hvm | tail -6

看到 `HOST Server listening...` 和 `PLUGIN_TAG ... checker ... RUNNING` 就说明环境起来了。

> 如果 `tmux` 没装，先执行 `!sudo apt-get update -qq && sudo apt-get install -y -qq tmux`。

## 3. 建立通信域

`mock-comm 112` 里的 `112` 是拓扑配置文件的名字，对应 `config/topo_meta/112.yaml`：
1 个超节点、1 个 server、2 个 rank。这一节的算例正好是 2 个 rank。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux send-keys -t hvm "$VM mock-comm 112" Enter
for i in $(seq 1 20); do
    tmux capture-pane -pt hvm | grep -q "config by 112.yaml" && break
    sleep 1
done

tmux capture-pane -pt hvm | tail -6
echo "--- data dir ---"
ls "$VM_INSTALL/data" | tr '\n' ' '

期望看到 `refresh comm config: file=112, server num=1`，并且 `data/` 里多出 **`ranktable.json`**。

这个文件记录了每个 rank 对应哪个 server、哪张卡，第 4 步就靠它。

## 4. 跑一次 ReduceScatter

用 `mpirun` 拉起 2 个进程，每个进程是一个 rank。参数含义：

| 参数 | 含义 |
| :--- | :--- |
| `-np 2` | 起 2 个 rank |
| `-b 64 -e 64` | 数据量从 64 字节到 64 字节（只跑一个尺寸） |
| `-d int32` | 数据类型 |
| `-o sum` | 归约方式：求和 |
| `-w 0 -n 1` | 预热 0 次、正式跑 1 次 |
| `-c 1` | 打开数值校验 |

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux send-keys -t hvm "mpirun --oversubscribe -np 2 $TEST_BIN/reduce_scatter_test -b 64 -e 64 -d int32 -o sum -w 0 -n 1 -c 1 > $VM_INSTALL/bin/log.txt 2>&1" Enter
for i in $(seq 1 30); do
    grep -q "check_result" "$VM_INSTALL/bin/log.txt" 2>/dev/null && break
    sleep 2
done

echo "--- log size ---"
ls -lh "$VM_INSTALL/bin/log.txt"
echo "--- result table ---"
grep -n -A 3 "check_result" "$VM_INSTALL/bin/log.txt"

## 5. 为什么数值校验显示 failed

你会看到这样一行：

```
data_size(Bytes): | aveg_time(us): | alg_bandwidth(GB/s): | check_result:
64                | 1000.00        | 0.00006              | failed
```

**这是正常的，不代表仿真失败。**

原因：HCCL-VM 是**任务图模拟器**，它记录并检查"谁给谁发了什么、依赖关系对不对"，
但**不真的做数值计算**。所以缓冲区里的数据不会被真正加起来，数值校验自然对不上。
`aveg_time` 恒等于 1000.00 us 也是同一个原因——那是占位值，不是真实耗时。

这一课真正看的是下一步的 **Checker**：它检查任务图正确性，这才是模拟器要验证的东西。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
grep -n "check result failed" "$VM_INSTALL/bin/log.txt" | head -3 || true
echo "--- op entries ---"
grep -o "Entry-Hccl[A-Za-z]*" "$VM_INSTALL/bin/log.txt" | sort | uniq -c

`Entry-HcclCommInitClusterInfo` 出现说明通信域初始化被调用；
如果连这个都没有，说明算例在初始化阶段就退出了，回到第 3 步检查 `ranktable.json` 是否生成。

## 6. 跑 Checker：真正的验收

Checker 读仿真过程中记录的任务图，检查三件事：任务依赖是否成环、内存访问是否越界、同步关系是否正确。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux send-keys -t hvm "$VM plugin run @checker" Enter
for i in $(seq 1 30); do
    tmux capture-pane -pt hvm | grep -q "CHECKER_RUN_SUMMARY" && break
    sleep 2
done

tmux capture-pane -pt hvm | tail -20

期望看到：

```
Checker result: commName=hccl_world_group, opIter=1, result=success
| hccl_world_group | 0 | 0 | success |
| hccl_world_group | 0 | 1 | success |
| big graph checker | success |
[CHECKER_RUN_SUMMARY] All Success (Total Op: 2, Big Graph: 1)
```

`Total Op: 2` 表示两个 rank 的算子都被 Checker 收到了。看到 `All Success` 就是本节通过。

## 7. 导出结果：实例关了就没了

实验环境的磁盘**不保证持久**：关机或重建实例后，写在实例上的文件可能全部消失。
所以这一步不是"存到某个目录"，而是**打包成一个文件，下载到你自己的电脑上**。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

CASE_DIR="$RECORD_ROOT/last"
mkdir -p "$CASE_DIR"
cp "$VM_INSTALL/bin/log.txt" "$CASE_DIR/test.log" 2>/dev/null
cp "$VM_INSTALL/data/ranktable.json" "$CASE_DIR/" 2>/dev/null
cp "$VM_INSTALL/data/topo.json" "$CASE_DIR/" 2>/dev/null

OUT="$HOME/hccl_vm_result_$(date +%Y%m%d_%H%M%S).tar.gz"
tar czf "$OUT" -C "$RECORD_ROOT" last
ls -lh "$OUT"
echo "exported: $OUT"

打包完成后，在左侧文件列表里找到 `hccl_vm_result_*.tar.gz`，**右键 → Download** 下载到本地。

三种留档方式的区别：

| 想留档 | 怎么做 | 结果 |
| :--- | :--- | :--- |
| 下载到电脑 | 左侧右键 Download | ✅ 关机后仍然保留 |
| 提交到仓库 | `git add` 后 push | ✅ 可长期共享 |
| 只放在实例上 | 什么都不做 | ❌ 关机即丢失 |

压缩包解开后，`last/test.log` 是完整运行日志，`last/ranktable.json` 是本次的 rank 与设备映射。

## 8. 遇到问题先看这里

| 现象 | 原因 | 处理 |
| :--- | :--- | :--- |
| `tmux: command not found` | 镜像里没装 tmux | `!sudo apt-get update -qq && sudo apt-get install -y -qq tmux` |
| 单元一直不返回 | 直接执行了 `start` | 用 `tmux kill-server` 清掉，重新按第 2 步用 tmux 起 |
| `get serverKey by rankId:0 failed` | 没做第 3 步，或 `ranktable.json` 没生成 | 重跑 `mock-comm 112`，确认 `data/ranktable.json` 存在 |
| `libxxx.so: cannot open` | 没有经过 `run_hccl_vm.sh`，库路径没设 | 用 `$VM` 或 `$PACK/run_hccl_vm.sh`，不要直接调 `bin/hccl-vm` |
| Checker 显示 `Total Op: 0` | 算子没执行到 | 先看第 4 步日志里有没有 `Entry-HcclCommInitClusterInfo` |
| 结束时想清理 | 会话还在后台 | `!tmux kill-server` |